In [1]:
import pandas as pd
import itertools
from nupack import *
import numpy as np
import math

In [3]:
#Function to generate all possible mutations given a starting sequence and the first and last nucleotide to be mutated
def generate_mutations(sequence, start, end):
    """
    Generate all possible mutations in a given range of a DNA sequence.
    
    Parameters
    ----------
    sequence : str
        Original DNA sequence (A, T, C, G)
    start : int
        Start position (1-based index, inclusive)
    end : int
        End position (1-based index, inclusive)
        
    Returns
    -------
    pd.DataFrame
        DataFrame with columns ['name', 'mutation']
    """
    bases = ['A', 'T', 'C', 'G']
    seq = list(sequence)
    region_indices = range(start - 1, end)
    results = []

    # Generate all combinations of mutations/deletions for each subset of positions
    for r in range(1, len(region_indices) + 1):
        for positions in itertools.combinations(region_indices, r):
            # For each position, choose one of the 4 possible bases + deletion
            options_per_pos = []
            for pos in positions:
                original = seq[pos]
                possible_changes = [b for b in bases if b != original] + ['']  # '' = deletion
                options_per_pos.append([(pos, change) for change in possible_changes])

            # Cartesian product of all mutation possibilities
            for mutation_combo in itertools.product(*options_per_pos):
                mutated_seq = seq.copy()
                # Apply mutations
                for pos, change in sorted(mutation_combo, reverse=True):
                    if change == '':
                        del mutated_seq[pos]  # deletion
                    else:
                        mutated_seq[pos] = change
                results.append(''.join(mutated_seq))

    # Remove duplicates and create DataFrame
    unique_results = sorted(set(results))
    df = pd.DataFrame({
        'Name': [f'Apt6_d5_{i+1}' for i in range(len(unique_results))],
        'Sequence': unique_results
    })
    return df

In [ ]:
#Starting Informations
sequence="ATAGTCCCTGGCGTGCTTGACAGCAACACGAACACTGAACGTTCTTAACAAGCCTGGCAGAGCAGGTACGGTGTCA"
start=67
end=76

In [10]:
df=generate_mutations(sequence, start, end)

In [ ]:
#Equilibrium pair probability matrix for the starting sequence
nupack_model = Model(material='dna04-nupack3', ensemble='some-nupack3', celsius=25, sodium=0.137, magnesium=0)
probability_matrix_starting_aptamer = pairs(strands=sequence, model=nupack_model)
probability_matrix_array_starting_aptamer=probability_matrix_starting_aptamer.to_array()

In [24]:
#From the Dataframe containing all the permutations compute, for each mutated permutation,
#the minimum SSD between every x square sub-matrix, where x is the length of the permutation,
#taken along the diagonal of the starting sequence’s equilibrium pair-probability matrix
#and the permutation’s equilibrium pair-probability matrix.
sd['SSD']=""
df['SSD Normalized']=""
for h in df:
    probability_matrix_mutation = pairs(strands=h, model=nupack_model)
    probability_matrix_array_mutation=probability_matrix_mutation.to_array()
    b=probability_matrix_array_mutation
    ssd=0
    i=len(b)
    x=0
    while(i < len(probability_matrix_array_starting_aptamer)+1):
        a=probability_matrix_array_starting_aptamer[x:i,x:i]
        ssd_cycle=np.sum((a - b) ** 2)
        if ssd_cycle < ssd or ssd==0:
            ssd =ssd_cycle
        i=i+1
        x=x+1
    ssd_normalized=math.sqrt((ssd)/(2*len(h)))
    index=Apt_6_d5_unique.index[df['Sequence'] == h]
    df.loc[index,"SSD"]=ssd
    df.loc[index,"SSD Normalized"]=1 - ssd_normalized
df.to_csv("AptX_all_combinations_SSD.csv")